In [1]:
import sys, torch
from google.colab import drive

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
assert torch.cuda.is_available(), "CHƯA BẬT GPU! Vào Runtime -> Change runtime type -> T4 GPU rồi chạy lại."
print("GPU    :", torch.cuda.get_device_name(0))
!nvidia-smi

drive.mount('/content/drive')

Python : 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA   : True
GPU    : NVIDIA L4
Fri Nov 20 13:37:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                        

In [2]:
import os, shutil, hashlib

DRIVE_TRAIN_PARQUET = "/content/drive/MyDrive/PTDLKD/train.parquet"
DRIVE_TASK10_DIR    = "/content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery"

# ====== SEED THỰC CHẠY (2024 theo kế hoạch Task 2; đổi 1024 nếu muốn chạy nguyên bản gốc) ======
RUN_SEED = 2024

# SHA-256 chuẩn từ biên bản Task 6 (run_info.md)
EXPECTED_TRAIN_SHA256 = "b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96"

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

# Dọn sạch thư mục/file tạm cũ (tránh trùng tên với package 'data' của repo)
for old_path in ["/content/data", "/content/frn_local_data"]:
    if os.path.isdir(old_path):
        shutil.rmtree(old_path)
    elif os.path.isfile(old_path):
        os.remove(old_path)

CACHE_DIR = "/content/frn_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_TASK10_DIR, "full"), exist_ok=True)

assert os.path.isfile(DRIVE_TRAIN_PARQUET), f"Không tìm thấy file: {DRIVE_TRAIN_PARQUET}"

LOCAL_TRAIN_PARQUET = os.path.join(CACHE_DIR, "train.parquet")
shutil.copy2(DRIVE_TRAIN_PARQUET, LOCAL_TRAIN_PARQUET)

actual_sha = sha256_file(LOCAL_TRAIN_PARQUET)
print("DRIVE_TRAIN_PARQUET:", DRIVE_TRAIN_PARQUET)
print("LOCAL_TRAIN_PARQUET:", LOCAL_TRAIN_PARQUET, f"({os.path.getsize(LOCAL_TRAIN_PARQUET):,} bytes)")
print("SHA-256 thực tế    :", actual_sha)
print("Khớp SHA-256 Task 6:", actual_sha == EXPECTED_TRAIN_SHA256)
assert actual_sha == EXPECTED_TRAIN_SHA256, "SHA-256 không khớp Task 6! Hãy kiểm tra lại đúng file train.parquet."

DRIVE_TRAIN_PARQUET: /content/drive/MyDrive/PTDLKD/train.parquet
LOCAL_TRAIN_PARQUET: /content/frn_cache/train.parquet (61,075,680 bytes)
SHA-256 thực tế    : b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96
Khớp SHA-256 Task 6: True


In [3]:
%%bash
set -e
cd /content
if [ ! -d "frn-50k-baseline" ]; then
    git clone https://github.com/Dingdong-Inc/frn-50k-baseline.git
fi
cd /content/frn-50k-baseline
git checkout 8af694a09a5a427d1d50648d348c022565a781f2
git log -n 1 --oneline

# KHÔNG pip install pypots (repo dùng bản pypots tùy biến nằm sẵn trong latent_demand_recovery/)
pip install -q datasets pygrinder tsdb einops sympy h5py tensorboard

ls -la /content/frn-50k-baseline/latent_demand_recovery

8af694a fix a bug due to .gitignore
total 16
drwxr-xr-x 4 root root 4096 Nov 20 13:26 .
drwxr-xr-x 5 root root 4096 Nov 20 13:26 ..
drwxr-xr-x 4 root root 4096 Nov 20 13:26 exp
drwxr-xr-x 8 root root 4096 Nov 20 13:27 pypots


HEAD is now at 8af694a fix a bug due to .gitignore


In [5]:
import os, sys, torch, numpy as np, pandas as pd

REPO_LDR_ROOT = "/content/frn-50k-baseline/latent_demand_recovery"
EXP_DIR       = os.path.join(REPO_LDR_ROOT, "exp")

print("Files in latent_demand_recovery    :", os.listdir(REPO_LDR_ROOT))
print("Files in latent_demand_recovery/exp:", os.listdir(EXP_DIR))

for mod in ["data", "model"]:
    sys.modules.pop(mod, None)
for p in [EXP_DIR, REPO_LDR_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pypots
from data import load_data
from model import load_model

print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("pypots :", pypots.__file__)
assert REPO_LDR_ROOT in pypots.__file__, "CẢNH BÁO: pypots đang không nạp từ thư mục repo!"

test_cfg = {
    'model': 'TimesNet',
    'saving_path': '/content/tmp_smoke_check',
    'EPOCHS': 5, 'batch_size': 128, 'patience': 5,
    'n_layers': 2, 'd_model': 64, 'd_ffn': 32,
    'n_heads': 4, 'd_k': 16, 'd_v': 16,
    'dropout': 0., 'attn_dropout': 0.,
    'lr': 0.001, 'weight_decay': 1e-5,
    'OT': 1, 'missing_rate': 0.0,
    'n_steps': 480, 'n_features': 6, 'patch_len': 16,
}
m = load_model(test_cfg)
print("Khởi tạo TimesNet thành công trên:", m.device)
print("Đối chiếu tham số: epochs=5, batch_size=128, lr=0.001, n_steps=480, n_features=6, top_k=7, n_kernels=5")

Files in latent_demand_recovery    : ['exp', 'pypots']
Files in latent_demand_recovery/exp: ['app.py', 'model', 'submit.sh', 'data']


2026-11-20 13:39:17 [INFO]: Using the given device: cuda:0
2026-11-20 13:39:17 [INFO]: Model files will be saved to /content/tmp_smoke_check/TimesNet/20261120_T133917
2026-11-20 13:39:17 [INFO]: Tensorboard file will be saved to /content/tmp_smoke_check/TimesNet/20261120_T133917/tensorboard


cuda:0
numpy  : 2.4.6
pandas : 2.2.3
torch  : 2.14.1+cu130 | CUDA: True
pypots : /content/frn-50k-baseline/latent_demand_recovery/pypots/__init__.py


2026-11-20 13:39:18 [INFO]: TimesNet initialized with the given hyperparameters, the number of trainable parameters: 1,354,438


Khởi tạo TimesNet thành công trên: cuda:0
Đối chiếu tham số: epochs=5, batch_size=128, lr=0.001, n_steps=480, n_features=6, top_k=7, n_kernels=5


In [6]:
import os, glob, subprocess

REPO_ROOT = "/content/frn-50k-baseline"
LDR_ROOT  = os.path.join(REPO_ROOT, "latent_demand_recovery")

# Reset sạch về bản gốc trước khi áp patch (an toàn khi chạy lại cell)
subprocess.run(["git", "-C", REPO_ROOT, "checkout", "--", "latent_demand_recovery"], check=True)

app_path = os.path.join(LDR_ROOT, "exp", "app.py")
gen_matches = glob.glob(os.path.join(LDR_ROOT, "**", "generate_data.py"), recursive=True)
assert len(gen_matches) == 1, f"Tìm thấy {len(gen_matches)} file generate_data.py: {gen_matches}"
gen_path = gen_matches[0]
print("Patch 1:", gen_path)
print("Patch 2:", app_path)

# --- 1. Patch generate_data.py ---
with open(gen_path, "r", encoding="utf-8") as f:
    gen_code = f.read()

old_gen_load = '''def load_data(CONFIG):
    dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
    data = dataset['train'].to_pandas()

    data = data.sort_values(by=['store_id', 'product_id', 'dt'])'''

new_gen_load = '''def load_data(CONFIG):
    local_parquet = os.environ.get("FRN_LOCAL_TRAIN_PARQUET", "")
    if local_parquet:
        print(f"[load_data] Reading local train parquet: {local_parquet}", flush=True)
        data = pd.read_parquet(local_parquet)
    else:
        dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
        data = dataset['train'].to_pandas()

    data = data.sort_values(by=['store_id', 'product_id', 'dt'], kind='mergesort').reset_index(drop=True)
    print(f"[load_data] Loaded train rows: {len(data)}, cols: {len(data.columns)}", flush=True)'''

assert old_gen_load in gen_code, "Không khớp đoạn load_data trong generate_data.py!"
gen_code = gen_code.replace(old_gen_load, new_gen_load, 1)
with open(gen_path, "w", encoding="utf-8") as f:
    f.write(gen_code)

# --- 2. Patch app.py ---
with open(app_path, "r", encoding="utf-8") as f:
    app_code = f.read()

old_app_recovery = '''def _demand_recovery(imputation):
    dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
    data = dataset['train'].to_pandas()
    data = data.sort_values(by=['store_id', 'product_id', 'dt'])'''

new_app_recovery = '''def _demand_recovery(imputation):
    local_parquet = os.environ.get("FRN_LOCAL_TRAIN_PARQUET", "")
    if local_parquet:
        print(f"[_demand_recovery] Reading local train parquet: {local_parquet}", flush=True)
        data = pd.read_parquet(local_parquet)
    else:
        dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
        data = dataset['train'].to_pandas()
    data = data.sort_values(by=['store_id', 'product_id', 'dt'], kind='mergesort').reset_index(drop=True)
    print(f"[_demand_recovery] Loaded train rows: {len(data)}, cols: {len(data.columns)}", flush=True)'''

assert old_app_recovery in app_code, "Không khớp đoạn _demand_recovery trong app.py!"
app_code = app_code.replace(old_app_recovery, new_app_recovery, 1)

old_save_pq = "    data.to_parquet(f'./demand/demand.parquet')"
new_save_pq = """    data.to_parquet(f'./demand/demand.parquet', index=False)
    data[['store_id', 'product_id', 'dt']].to_parquet(f'./demand/key_order.parquet', index=False)
    print(f"[_demand_recovery] Saved ./demand/demand.parquet ({len(data)} rows, {len(data.columns)} cols)", flush=True)"""
assert old_save_pq in app_code, "Không khớp đoạn to_parquet trong app.py!"
app_code = app_code.replace(old_save_pq, new_save_pq, 1)

old_call_rec = """def demand_recovery(CONFIG):
    imputation = _imputation(CONFIG)
    demand_df = None
    if CONFIG['missing_rate'] == 0:
        demand_df = _demand_recovery(imputation)
        evaluation_decoupling(demand_df)
    return demand_df"""

new_call_rec = """def demand_recovery(CONFIG):
    import gc
    imputation = _imputation(CONFIG)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    demand_df = None
    if CONFIG['missing_rate'] == 0:
        demand_df = _demand_recovery(imputation)
        evaluation_decoupling(demand_df)
    return demand_df"""
assert old_call_rec in app_code, "Không khớp hàm demand_recovery trong app.py!"
app_code = app_code.replace(old_call_rec, new_call_rec, 1)

old_seed_call = "    set_seed(seed_value=1024)"
new_seed_call = f"""    print("ACTIVE_SEED =", {RUN_SEED}, flush=True)
    print("ACTIVE_CONFIG =", CONFIG, flush=True)
    set_seed(seed_value={RUN_SEED})"""
assert old_seed_call in app_code, "Không khớp dòng set_seed(seed_value=1024) trong app.py!"
app_code = app_code.replace(old_seed_call, new_seed_call, 1)

with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

diff_text = subprocess.check_output(["git", "-C", REPO_ROOT, "diff"], text=True)
diff_path = os.path.join(DRIVE_TASK10_DIR, "code_changes.diff")
with open(diff_path, "w", encoding="utf-8") as f:
    f.write(diff_text)

print("=== ĐÃ LƯU CODE DIFF TẠI:", diff_path, "===")
print(diff_text)

Patch 1: /content/frn-50k-baseline/latent_demand_recovery/exp/data/generate_data.py
Patch 2: /content/frn-50k-baseline/latent_demand_recovery/exp/app.py
=== ĐÃ LƯU CODE DIFF TẠI: /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/code_changes.diff ===
diff --git a/latent_demand_recovery/exp/app.py b/latent_demand_recovery/exp/app.py
index 400e501..8e387cd 100755
--- a/latent_demand_recovery/exp/app.py
+++ b/latent_demand_recovery/exp/app.py
@@ -63,9 +63,15 @@ def _imputation(CONFIG):
     return imputation
     
 def _demand_recovery(imputation):
-    dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
-    data = dataset['train'].to_pandas()
-    data = data.sort_values(by=['store_id', 'product_id', 'dt'])
+    local_parquet = os.environ.get("FRN_LOCAL_TRAIN_PARQUET", "")
+    if local_parquet:
+        print(f"[_demand_recovery] Reading local train parquet: {local_parquet}", flush=True)
+        data = pd.read_parquet(local_parquet)
+    else:
+        dataset = load_datase

In [7]:
import os, time, subprocess

EXP_DIR = "/content/frn-50k-baseline/latent_demand_recovery/exp"
DRIVE_LOG_PATH = os.path.join(DRIVE_TASK10_DIR, "full", "timesnet_recovery.log")

# Dọn sạch thư mục output cũ trong /content (nếu có)
!rm -rf /content/frn-50k-baseline/latent_demand_recovery/exp/demand
!rm -rf /content/frn-50k-baseline/latent_demand_recovery/exp/save

env = os.environ.copy()
env["FRN_LOCAL_TRAIN_PARQUET"] = LOCAL_TRAIN_PARQUET
env["PYTHONUNBUFFERED"] = "1"

cmd = ["python", "-u", "app.py", "--model", "TimesNet", "--missing_rate", "0"]
print("Running command:", " ".join(cmd))
print("Working dir    :", EXP_DIR)
print("Log file       :", DRIVE_LOG_PATH)

t0 = time.time()
with open(DRIVE_LOG_PATH, "w", encoding="utf-8") as log_f:
    proc = subprocess.Popen(
        cmd,
        cwd=EXP_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
        log_f.write(line)
        log_f.flush()
    ret = proc.wait()

elapsed_sec = time.time() - t0
print(f"\n=== KẾT THÚC | Exit code: {ret} | Thời gian: {elapsed_sec/60:.2f} phút ===")
assert ret == 0, f"Lệnh app.py thoát với mã lỗi {ret}! Hãy kiểm tra log ở trên."

Running command: python -u app.py --model TimesNet --missing_rate 0
Working dir    : /content/frn-50k-baseline/latent_demand_recovery/exp
Log file       : /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/timesnet_recovery.log
I0000 00:00:1795181991.060191    5361 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795181993.474675    5361 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795181993.475283    5361 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
cuda:0
Namespace(model='TimesNet', missing_rate=0.0)
ACTIVE_SEED = 2024
ACTIV

In [8]:
import os, shutil, glob

EXP_DIR = "/content/frn-50k-baseline/latent_demand_recovery/exp"
SRC_DEMAND_PQ  = os.path.join(EXP_DIR, "demand", "demand.parquet")
SRC_KEY_ORDER  = os.path.join(EXP_DIR, "demand", "key_order.parquet")
SRC_IMPUTE_NPY = os.path.join(EXP_DIR, "demand", "TimesNet_imputation_0.0.npy")
SRC_SAVE_DIR   = os.path.join(EXP_DIR, "save", "TimesNet")

FULL_DRIVE_DIR = os.path.join(DRIVE_TASK10_DIR, "full")
os.makedirs(FULL_DRIVE_DIR, exist_ok=True)

# 1. File chính bàn giao cho Task 11
DRIVE_DEMAND_PQ = os.path.join(FULL_DRIVE_DIR, "demand_timesnet.parquet")
assert os.path.exists(SRC_DEMAND_PQ), "Không tìm thấy ./demand/demand.parquet!"
shutil.copy2(SRC_DEMAND_PQ, DRIVE_DEMAND_PQ)
print(f"Đã lưu file chính: {DRIVE_DEMAND_PQ} ({os.path.getsize(DRIVE_DEMAND_PQ):,} bytes)")

# 2. Lưu kèm file imputation theo giờ + thứ tự khóa
if os.path.exists(SRC_IMPUTE_NPY):
    dst_npy = os.path.join(FULL_DRIVE_DIR, "TimesNet_imputation_0.0.npy")
    shutil.copy2(SRC_IMPUTE_NPY, dst_npy)
    print(f"Đã lưu imputation theo giờ: {dst_npy} ({os.path.getsize(dst_npy):,} bytes)")

if os.path.exists(SRC_KEY_ORDER):
    dst_key = os.path.join(FULL_DRIVE_DIR, "key_order.parquet")
    shutil.copy2(SRC_KEY_ORDER, dst_key)
    print(f"Đã lưu thứ tự khóa: {dst_key}")

# 3. Lưu checkpoint nếu có
DRIVE_CKPT_DIR = os.path.join(FULL_DRIVE_DIR, "checkpoint_TimesNet")
if os.path.exists(SRC_SAVE_DIR):
    if os.path.exists(DRIVE_CKPT_DIR):
        shutil.rmtree(DRIVE_CKPT_DIR)
    shutil.copytree(SRC_SAVE_DIR, DRIVE_CKPT_DIR)
    print("Đã sao lưu checkpoint sang:", DRIVE_CKPT_DIR)
    for cf in glob.glob(os.path.join(DRIVE_CKPT_DIR, "**", "*"), recursive=True):
        if os.path.isfile(cf):
            print("  -", cf, f"({os.path.getsize(cf):,} bytes)")
else:
    print("Không thấy thư mục checkpoint ./save/TimesNet.")

Đã lưu file chính: /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/demand_timesnet.parquet (81,562,029 bytes)
Đã lưu imputation theo giờ: /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/TimesNet_imputation_0.0.npy (288,000,128 bytes)
Đã lưu thứ tự khóa: /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/key_order.parquet
Đã sao lưu checkpoint sang: /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/checkpoint_TimesNet
  - /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/checkpoint_TimesNet/20261120_T134010/TimesNet.pypots (6,237,441 bytes)
  - /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/full/checkpoint_TimesNet/20261120_T134010/tensorboard/events.out.tfevents.1795182010.bf0adbe69a06.5361.0.pypots (304,681 bytes)


In [9]:
import os, numpy as np, pandas as pd

KEY_COLS = ["store_id", "product_id", "dt"]
EXPECTED_ROWS = 4_500_000
TARGET_COL = "sale_amount_pred"
CHECK_CSV_PATH = os.path.join(DRIVE_TASK10_DIR, "demand_file_check.csv")

df_train = pd.read_parquet(LOCAL_TRAIN_PARQUET)
df_demand = pd.read_parquet(DRIVE_DEMAND_PQ)

train_cols = list(df_train.columns)
demand_cols = list(df_demand.columns)
missing_orig_cols = [c for c in train_cols if c not in demand_cols]
extra_cols = [c for c in demand_cols if c not in train_cols]

print("SHA-256 demand_timesnet.parquet:", sha256_file(DRIVE_DEMAND_PQ))
print(f"Train shape : {df_train.shape}")
print(f"Demand shape: {df_demand.shape}")
print("Missing original cols:", missing_orig_cols)
print("Extra cols           :", extra_cols)

assert len(df_demand) == EXPECTED_ROWS, f"Sai số dòng: {len(df_demand)}"
assert len(train_cols) == 19, f"Tập train không phải 19 cột: {len(train_cols)}"
assert missing_orig_cols == [] and extra_cols == [TARGET_COL], "Schema cột không khớp!"

# Kiểm tra khóa duy nhất & trùng khớp hoàn toàn tập train
n_dup_keys = int(df_demand.duplicated(subset=KEY_COLS).sum())
train_idx = pd.MultiIndex.from_frame(df_train[KEY_COLS])
demand_idx = pd.MultiIndex.from_frame(df_demand[KEY_COLS])
n_missing_keys = int(len(train_idx.difference(demand_idx)))
n_extra_keys = int(len(demand_idx.difference(train_idx)))

# Kiểm tra chất lượng sale_amount_pred
pred = pd.to_numeric(df_demand[TARGET_COL], errors="coerce")
n_pred_missing = int(pred.isna().sum())
n_pred_non_finite = int((~np.isfinite(pred.dropna())).sum())
n_pred_negative = int((pred.dropna() < 0).sum())

# Sanity check: thứ tự khóa khớp train đã sort; sale_amount & hours_sale gốc không bị sửa đổi
df_train_sorted = df_train.sort_values(KEY_COLS, kind="mergesort").reset_index(drop=True)
assert df_train_sorted[KEY_COLS].equals(df_demand[KEY_COLS]), "Thứ tự khóa không khớp train đã sort!"
max_sale_diff = float((df_train_sorted["sale_amount"] - df_demand["sale_amount"]).abs().max())
sample_hours_equal = all(
    np.array_equal(a, b)
    for a, b in zip(df_train_sorted["hours_sale"].iloc[:1000], df_demand["hours_sale"].iloc[:1000])
)
print(f"Max |sale_amount gốc - demand|: {max_sale_diff}")
print(f"hours_sale (1000 dòng mẫu) giữ nguyên quan sát gốc: {sample_hours_equal}")
print("\nThống kê sale_amount vs sale_amount_pred:")
print(df_demand[["sale_amount", TARGET_COL]].describe())

check_df = pd.DataFrame([{
    "file_path": DRIVE_DEMAND_PQ,
    "n_rows": len(df_demand),
    "n_cols": len(demand_cols),
    "n_duplicate_keys": n_dup_keys,
    "n_missing_keys_vs_train": n_missing_keys,
    "n_extra_keys_vs_train": n_extra_keys,
    "n_pred_missing": n_pred_missing,
    "n_pred_non_finite": n_pred_non_finite,
    "n_pred_negative": n_pred_negative,
}])
check_df.to_csv(CHECK_CSV_PATH, index=False)
display(check_df)

error_cols = [
    "n_duplicate_keys",
    "n_missing_keys_vs_train",
    "n_extra_keys_vs_train",
    "n_pred_missing",
    "n_pred_non_finite",
    "n_pred_negative",
]
assert (check_df[error_cols].iloc[0] == 0).all(), "PHÁT HIỆN LỖI ĐỐI SOÁT! Không bàn giao file này."
assert max_sale_diff == 0.0 and sample_hours_equal, "Dữ liệu cột gốc bị thay đổi!"
print("PASSED: demand_timesnet.parquet đạt đủ mọi tiêu chí kiểm định (toàn bộ bộ đếm lỗi = 0).")

SHA-256 demand_timesnet.parquet: c34f4eafe087d225865a624ef24d213af566496541458c3075f6e596d9f1d694
Train shape : (4500000, 19)
Demand shape: (4500000, 20)
Missing original cols: []
Extra cols           : ['sale_amount_pred']
Max |sale_amount gốc - demand|: 0.0
hours_sale (1000 dòng mẫu) giữ nguyên quan sát gốc: True

Thống kê sale_amount vs sale_amount_pred:
        sale_amount  sale_amount_pred
count  4.500000e+06      4.500000e+06
mean   9.985913e-01      1.206512e+00
std    1.406738e+00      1.679486e+00
min    0.000000e+00      0.000000e+00
25%    4.000000e-01      5.000000e-01
50%    7.000000e-01      8.000000e-01
75%    1.100000e+00      1.343607e+00
max    4.490000e+01      5.009378e+01


,file_path,n_rows,n_cols,n_duplicate_keys,n_missing_keys_vs_train,n_extra_keys_vs_train,n_pred_missing,n_pred_non_finite,n_pred_negative
0,/content/drive/MyDrive/PTDLKD/Task10_TimesNet_...,4500000,20,0,0,0,0,0,0


PASSED: demand_timesnet.parquet đạt đủ mọi tiêu chí kiểm định (toàn bộ bộ đếm lỗi = 0).


In [10]:
import os, re, pandas as pd

DECOUPLING_CSV_PATH = os.path.join(DRIVE_TASK10_DIR, "decoupling_metrics.csv")
LOG_REL_PATH = "full/timesnet_recovery.log"

with open(DRIVE_LOG_PATH, "r", encoding="utf-8") as f:
    log_text = f.read()

print("Xác nhận missing_rate=0.0 trong log:", "missing_rate=0.0" in log_text)

# Lấy đúng giá trị decoupling score in từ print(metric), giữ nguyên giá trị gốc (không đổi sang %)
m_raw  = re.findall(r"\bsale_amount\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", log_text)
m_pred = re.findall(r"\bsale_amount_pred\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", log_text)

assert len(m_raw) >= 1 and len(m_pred) >= 1, "Không tìm thấy decoupling score trong log!"

raw_score_str = m_raw[-1]
timesnet_score_str = m_pred[-1]

decoupling_df = pd.DataFrame([
    {"method": "Raw Sales", "decoupling_score": raw_score_str,      "log_path": LOG_REL_PATH},
    {"method": "TimesNet",  "decoupling_score": timesnet_score_str, "log_path": LOG_REL_PATH},
])
decoupling_df.to_csv(DECOUPLING_CSV_PATH, index=False)
display(decoupling_df)

Xác nhận missing_rate=0.0 trong log: True


,method,decoupling_score,log_path
0,Raw Sales,-0.566447,full/timesnet_recovery.log
1,TimesNet,0.069918,full/timesnet_recovery.log


In [12]:
import os, sys, torch
from datetime import datetime, timezone

DEMAND_DRIVE_LINK = "https://drive.google.com/file/d/1f9db-U9f9C9OFS-V2EcmQv4FIU3bguR3/view?usp=sharing"
CKPT_DRIVE_LINK   = "https://drive.google.com/drive/folders/18D53nPab5toFuyuAQ00rRBrG_86WKKkR?usp=drive_link"

RUN_INFO_PATH = os.path.join(DRIVE_TASK10_DIR, "run_info.md")
demand_sha = sha256_file(DRIVE_DEMAND_PQ)
demand_size = os.path.getsize(DRIVE_DEMAND_PQ)
seed_note = (
    "Chạy theo kế hoạch Task 2 (đã sửa `set_seed(seed_value=2024)` trong bản sao `app.py`, xem `code_changes.diff`)."
    if RUN_SEED == 2024
    else "Chạy nguyên bản baseline gốc (`set_seed(seed_value=1024)`)."
)

FENCE = "`" * 3
lines = [
    "# Task 10 — Run Info (TimesNet Latent Demand Recovery)",
    "",
    f"- **Thời điểm hoàn tất (UTC):** {datetime.now(timezone.utc).isoformat()}",
    "- **Người thực hiện:** TV5",
    f"- **Môi trường chạy:** Google Colab | GPU: `{torch.cuda.get_device_name(0)}` | Python `{sys.version.split()[0]}` | PyTorch `{torch.__version__}`",
    "- **Mã tác giả đối chiếu (`frn-50k-baseline`):** commit `8af694a09a5a427d1d50648d348c022565a781f2`",
    f"- **Thay đổi mã nguồn trên bản sao:** Xem `code_changes.diff` (đồng bộ đọc `train.parquet` local + sắp xếp `['store_id', 'product_id', 'dt']` cho cả `load_data` và `_demand_recovery`, đặt seed `{RUN_SEED}`).",
    "",
    "## Lệnh chạy chính thức",
    f"{FENCE}bash",
    "cd latent_demand_recovery/exp",
    f"FRN_LOCAL_TRAIN_PARQUET={LOCAL_TRAIN_PARQUET} python -u app.py --model TimesNet --missing_rate 0",
    FENCE,
    "",
    "## Cấu hình thực tế",
    "- **Model:** `TimesNet`",
    "- **Missing rate:** `0.0` (chỉ khôi phục tại giờ hết hàng thực tế; `valid_idx.sum() == 0`; không tạo điểm thiếu nhân tạo; không dùng tập `eval`)",
    f"- **Seed thực dùng:** `{RUN_SEED}` — {seed_note}",
    "- **Siêu tham số baseline:** `epochs=5`, `batch_size=128`, `lr=0.001`, `weight_decay=1e-5`, `patience=5`, `n_layers=2`, `d_model=64`, `d_ffn=32`, `top_k=7`, `n_kernels=5`, `dropout=0.0`, `apply_nonstationary_norm=True`, `OT=1`",
    f"- **Dữ liệu đầu vào:** `train.parquet` (`4.500.000` dòng × `19` cột, SHA-256: `{EXPECTED_TRAIN_SHA256}` — khớp Task 6)",
    "- **Đầu vào TimesNet:** `(150000, 480, 6)` — 6 đặc trưng (`hours_sale`, `discount`, `holiday_flag`, `precpt`, `avg_temperature`, `hour_index/15`), mỗi cặp 90 ngày chia 3 đoạn × 30 ngày × 16 giờ (`6..21`).",
    f"- **Thời gian chạy:** `{elapsed_sec/60:.2f}` phút",
    "",
    "## Kết quả chính",
    f"- **Decoupling score (Raw Sales — `sale_amount`):** `{raw_score_str}`",
    f"- **Decoupling score (TimesNet — `sale_amount_pred`):** `{timesnet_score_str}`",
    "- **Log đầy đủ:** `full/timesnet_recovery.log`",
    "- **Đối soát file nhu cầu:** Xem `demand_file_check.csv` (`4.500.000` dòng × `20` cột, tất cả số đếm lỗi = `0`).",
    "",
    "## Liên kết tệp lớn trên Google Drive (mở quyền cho Task 11)",
    f"- **File chính `demand_timesnet.parquet`** (`{demand_size:,}` bytes | SHA-256: `{demand_sha}`):",
    f"  - Đường dẫn Drive: `{DRIVE_DEMAND_PQ}`",
    f"  - Link chia sẻ Drive: {DEMAND_DRIVE_LINK}",
    "- **Checkpoint TimesNet (`save/TimesNet/`):**",
    f"  - Đường dẫn Drive: `{DRIVE_CKPT_DIR}`",
    f"  - Link chia sẻ Drive: {CKPT_DRIVE_LINK}",
    "- **Mảng ước lượng theo giờ & thứ tự khóa:** `TimesNet_imputation_0.0.npy` + `key_order.parquet` (sắp xếp tăng dần theo `store_id`, `product_id`, `dt`) nằm trong cùng thư mục `full/`.",
]

run_info_md = "\n".join(lines) + "\n"
with open(RUN_INFO_PATH, "w", encoding="utf-8") as f:
    f.write(run_info_md)

print("Đã tạo:", RUN_INFO_PATH)
print(run_info_md)

Đã tạo: /content/drive/MyDrive/PTDLKD/Task10_TimesNet_Recovery/run_info.md
# Task 10 — Run Info (TimesNet Latent Demand Recovery)

- **Thời điểm hoàn tất (UTC):** 2026-11-20T15:30:56.214935+00:00
- **Người thực hiện:** TV5
- **Môi trường chạy:** Google Colab | GPU: `NVIDIA L4` | Python `3.13.16` | PyTorch `2.14.1+cu130`
- **Mã tác giả đối chiếu (`frn-50k-baseline`):** commit `8af694a09a5a427d1d50648d348c022565a781f2`
- **Thay đổi mã nguồn trên bản sao:** Xem `code_changes.diff` (đồng bộ đọc `train.parquet` local + sắp xếp `['store_id', 'product_id', 'dt']` cho cả `load_data` và `_demand_recovery`, đặt seed `2024`).

## Lệnh chạy chính thức
```bash
cd latent_demand_recovery/exp
FRN_LOCAL_TRAIN_PARQUET=/content/frn_cache/train.parquet python -u app.py --model TimesNet --missing_rate 0
```

## Cấu hình thực tế
- **Model:** `TimesNet`
- **Missing rate:** `0.0` (chỉ khôi phục tại giờ hết hàng thực tế; `valid_idx.sum() == 0`; không tạo điểm thiếu nhân tạo; không dùng tập `eval`)
- **Seed thự